## Imports


In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr


In [ ]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

MODEL = "gpt-4.1-mini"
openai = OpenAI()

# As an alternative, if you'd like to use Ollama instead of OpenAI
# Check that Ollama is running for you locally (see week1/day2 exercise) then uncomment these next 2 lines
# MODEL = "llama3.2"
# openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


In [ ]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""


### Defining the tools


In [ ]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            # destination_city is the input (parameter) of the function
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False,
    },
}

set_price_function = {
    "name": "set_ticket_price",
    "description": "Set the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
            "price": {
                "type": "number",
                "description": "The ticket price to set for the destination city in SGD",
            },
        },
        "required": ["destination_city", "price"],
        "additionalProperties": False,
    },
}


In [ ]:
# And this is included in a list of tools (wrapped in another json):

tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": set_price_function},
]


### The actual tools functions and also setting up the sql


In [ ]:
import sqlite3


In [ ]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute(
        "CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)"  # SQL query
    )
    conn.commit()


In [ ]:
def get_ticket_price(destination_city):
    print(
        f"DATABASE TOOL CALLED: Getting price for {destination_city}", flush=True
    )  # flush=true means print the statement ones immediately without waiting.
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            "SELECT price FROM prices WHERE city = ?", (destination_city.lower(),)
        )  # we are just appending the destination_city name so we don't get SQL injection attacks
        result = cursor.fetchone()
        return (
            f"Ticket price to {destination_city} is ${result[0]}"
            if result
            else "No price data available for this city"
        )


In [ ]:
def set_ticket_price(destination_city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            "INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?",
            (destination_city.lower(), price, price),
        )
        conn.commit()
        return f"Ticket price to {destination_city} has been set to ${price}"


In [ ]:
ticket_prices = {
    "london": 799,
    "paris": 899,
    "tokyo": 1420,
    "sydney": 2999,
    "singapore": 999,
}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)


In [ ]:
get_ticket_price("Tokyo")


In [ ]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )
    response = openai.chat.completions.create(
        model=MODEL, messages=messages, tools=tools
    )

    while (
        response.choices[0].finish_reason == "tool_calls"
    ):  # note the previous version is 'if', just change it to 'while' to allow it to loop through all the tool calls. -> handling multiple tool calls one after another. continually call the tools while it needs the tool calls. (unlikely to go on forever, LLMs don't really do that forever...)
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)

        for message in messages:
            print(message)

        response = openai.chat.completions.create(
            model=MODEL, messages=messages, tools=tools
        )

    return response.choices[0].message.content


In [ ]:
TOOLS_FUNCTIONS = {
    "get_ticket_price": get_ticket_price,
    "set_ticket_price": set_ticket_price,
}

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name not in TOOLS_FUNCTIONS.keys():
            responses.append(
                {
                    "role": "tool",
                    "content": f"Function {tool_call.function.name} not found",
                    "tool_call_id": tool_call.id,
                }
            )
            continue

        # get the args
        arguments = json.loads(
            tool_call.function.arguments
        )  # this arguments is a dictionary
        tool_call_result = TOOLS_FUNCTIONS[tool_call.function.name](**arguments)
        # here we are just calling the function directly.
        # 1 * unpacks a list (or other iterable like a tuple) into positional arguments, 2 ** unpacks a dictionary (so its keyword arguments), it's like javascript's spread operator !!! (so whatever arguments it has we will feed it to the function, so it works for ANY function.)
        # but make sure that the arguments match directly, like the function definition should also expect a parameter named 'destination_city' since our arguments contain a key called 'destination_city'
        print("this is working")
        responses.append(
            {"role": "tool", "content": tool_call_result, "tool_call_id": tool_call.id}
        )
    return responses


In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()
